# Bloco B - Rotulagem, Clusterização e Agregações

## 1. Carregar a tabela

In [0]:
df = spark.table("workspace.senhas.senhas_features")
print(f"Registros (14M): {df.count():,}")

Registros (14M): 14,344,371


## 2. Rotulagem por regras manuais (Spark — 14M)
Lógica da regra manual: tamanho < 6 ou só um tipo de caractere = fraca; tamanho >= 10 e 3+ tipos = forte; caso contrário, moderada.

In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "tipos",
    col("tem_maiuscula") + col("tem_minuscula") + col("tem_numero") + col("tem_especial")
)

df = df.withColumn(
    "forca_regras",
    when((col("tamanho") < 6) | (col("tipos") == 1), "fraca")
    .when((col("tamanho") >= 10) & (col("tipos") >= 3), "forte")
    .otherwise("moderada")
)

print("Distribuicao por regras manuais (14M):")
df.groupBy("forca_regras").count().orderBy("count", ascending=False).show()

Distribuicao por regras manuais (14M):
+------------+--------+
|forca_regras|   count|
+------------+--------+
|    moderada|11745989|
|       forte| 2317706|
|       fraca|  280676|
+------------+--------+



## 3. Amostra representativa para Machine Learning

A partir daqui, trabalhamos com uma amostra de 500 mil

In [0]:
FEATURES_CLUSTER = [
    "tamanho", "tem_maiuscula", "tem_minuscula", "tem_numero", "tem_especial",
    "qtd_maiusculas", "qtd_minusculas", "qtd_numeros", "qtd_especiais"
]

# Trazer amostra representativa para pandas (inclui a rotulagem por regras)
pdf = (df.select(*FEATURES_CLUSTER, "forca_regras")
         .sample(fraction=0.04, seed=42)
         .limit(500_000)
         .toPandas())

print(f"Amostra para ML: {len(pdf):,} registros")

Amostra para ML: 500,000 registros


## 4. Clusterização K-Means (scikit-learn)

In [0]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

# Normalizar e treinar
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(pdf[FEATURES_CLUSTER])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
pdf["cluster"] = kmeans.fit_predict(X_scaled)

# Mapear clusters para força pelo tamanho médio
media_tamanho = pdf.groupby("cluster")["tamanho"].mean().sort_values()
mapa_forca = {
    media_tamanho.index[0]: "fraca",
    media_tamanho.index[1]: "moderada",
    media_tamanho.index[2]: "forte"
}
pdf["forca_kmeans"] = pdf["cluster"].map(mapa_forca)

print("Mapa de clusters:", mapa_forca)
print("\nDistribuicao por K-Means (amostra):")
print(pdf["forca_kmeans"].value_counts())

Exception ignored on calling ctypes callback function: <function _ThreadpoolInfo._find_modules_with_dl_iterate_phdr.<locals>.match_module_callback at 0xfff172e9e5c0>
Traceback (most recent call last):
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 400, in match_module_callback
    self._make_module_from_path(filepath)
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 515, in _make_module_from_path
    module = module_class(filepath, prefix, user_api, internal_api)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 606, in __init__
    self.version = self.get_version()
                   ^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 646, in get_version
    config = get_config().split()
             ^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'split'
Ex

Mapa de clusters: {2: 'fraca', 1: 'moderada', 0: 'forte'}

Distribuicao por K-Means (amostra):
forte       241444
moderada    160880
fraca        97676
Name: forca_kmeans, dtype: int64


## 5. Comparativo entre os métodos

In [0]:
pdf["concordam"] = pdf["forca_regras"] == pdf["forca_kmeans"]

total = len(pdf)
concordam = int(pdf["concordam"].sum())
divergem = total - concordam

print(f"Concordam : {concordam:,} ({concordam/total*100:.1f}%)")
print(f"Divergem  : {divergem:,} ({divergem/total*100:.1f}%)")

Concordam : 228,461 (45.7%)
Divergem  : 271,539 (54.3%)


## 6. Comparativo cruzado (matriz Regras x K-Means)

In [0]:
import pandas as pd
comparativo = pd.crosstab(pdf["forca_regras"], pdf["forca_kmeans"])
comparativo = comparativo.reindex(index=["fraca","moderada","forte"],
                                  columns=["fraca","moderada","forte"])
print(comparativo)

forca_kmeans  fraca  moderada   forte
forca_regras                         
fraca          2134      6103    1996
moderada      92101    152521  165642
forte          3441      2256   73806


## 7. Persistir resultado

In [0]:
# Converter a amostra rotulada de volta para Spark e salvar como Delta
spark_df = spark.createDataFrame(pdf)
spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.senhas.senhas_rotuladas")

print("Tabela 'workspace.senhas.senhas_rotuladas' criada.")
print(f"Registros: {spark.table('workspace.senhas.senhas_rotuladas').count():,}")

Tabela 'workspace.senhas.senhas_rotuladas' criada.
Registros: 500,000
